In [0]:
bronze_path = "/Volumes/medallion/bronze/bronze_data"
silver_path = "/Volumes/medallion/silver/silver_data"
gold_path   = "/Volumes/medallion/gold/gold_data"

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

In [0]:
df_raw=spark.createDataFrame(data, columns)

### BRONZE LAYER

In [0]:
df_raw.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

### SILVER LAYER

In [0]:
df = spark.read.format("delta").load(bronze_path)

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### type casting

In [0]:
df = df.withColumn("amount", df.amount.cast("int"))
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- quantity: long (nullable = true)



### handle nulls

In [0]:
df.filter(df.amount.isNull() | df.city.isNull()).display()

order_id,customer_id,product,category,city,date,amount,quantity
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
107,C005,Headphones,Accessories,null,2024-01-08,3000,1


In [0]:
df=df.fillna({"amount":0, "city":"unknown"})
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
# Convert negative amounts to positive
df = df.withColumn("amount", when(df.amount < 0, -df.amount).otherwise(df.amount))
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
# Convert date column from string to date type
df = df.withColumn("date", to_date(df.date, "yyyy-MM-dd"))

### Remove Duplicates

In [0]:
# Remove duplicate rows
df = df.dropDuplicates()
print(f"Rows after removing duplicates: {df.count()}")
df.display()

Rows after removing duplicates: 10


order_id,customer_id,product,category,city,date,amount,quantity
108,C006,Laptop,Electronics,Bangalore,2024-01-09,45000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


###Keep latest record based on date

In [0]:
# Keep only the latest record for each order based on date
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())
df = df.withColumn("row_num", row_number().over(window_spec)) \
       .filter(col("row_num") == 1) \
       .drop("row_num")

print(f"Rows after keeping latest records: {df.count()}")
df.display()

Rows after keeping latest records: 9


order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### overwrite to silver layer

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

### GOLD LAYER
read from silver layer


In [0]:
df_sil = spark.read.format("delta").load(silver_path)

Total sales by product

In [0]:
product_sales = df_sil.groupBy("product") \
    .agg(sum("amount").alias("total_sales"))
product_sales.display()

product,total_sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,150000
Headphones,3000


In [0]:
#Total sales by category
category_sales = df_sil.groupBy("category") \
    .agg(sum("amount").alias("total_sales"))
category_sales.display()

category,total_sales
Electronics,205000
Accessories,11000


In [0]:
#total sales by city
city_sales=df_sil.groupBy("city")\
    .agg(sum("amount").alias("total_sales"))
city_sales.display()

city,total_sales
Delhi,55000
Chennai,33000
Hyderabad,72000
Bangalore,45000
Mumbai,8000
unknown,3000


### customer metrics

In [0]:
#Total orders per customer
customer_metrics = df.groupBy("customer_id") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("amount").alias("total_spent")
    )
customer_metrics.display()

customer_id,total_orders,total_spent
C001,2,72000
C002,2,18000
C003,1,55000
C004,1,8000
C005,1,3000
C006,1,45000
C007,1,15000


### top alysis

In [0]:
# Top selling product
top_product = product_sales.orderBy(col("total_sales").desc()).limit(1)
print("Top Selling Product:")
top_product.display()

Top Selling Product:


product,total_sales
Laptop,150000


In [0]:
# Top customer by total spent
top_customer = customer_metrics.orderBy(col("total_spent").desc()).limit(1)
print("\nTop Customer:")
top_customer.display()


Top Customer:


customer_id,total_orders,total_spent
C001,2,72000


### uploading to gold schema

In [0]:
product_sales.write.format("parquet").mode("overwrite").save(gold_path + "/product_sales")
category_sales.write.format("parquet").mode("overwrite").save(gold_path + "/category_sales")
city_sales.write.format("parquet").mode("overwrite").save(gold_path + "/city_sales")
customer_metrics.write.format("parquet").mode("overwrite").save(gold_path + "/customer_spend")
top_product.write.format("parquet").mode("overwrite").save(gold_path + "/top_product")
top_customer.write.format("parquet").mode("overwrite").save(gold_path + "/top_customer")